In [193]:
from datasets import load_dataset
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel
from torch.nn.utils.rnn import pad_sequence

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

### How the data will flow in the project 

Load dataset

        ↓
Explore the data

        ↓

Train/Validation/Test split

        ↓

Build vocabulary (using ONLY the training set)

        ↓

Tokenize the text

        ↓

Convert tokens to integer IDs

        ↓

Pad/Truncate sequences

        ↓

Convert to PyTorch tensors

        ↓

Create a custom Dataset

        ↓

Create a DataLoader

        ↓

Build the LSTM model

        ↓

Train

        ↓

Evaluate

In [194]:
data = load_dataset("fancyzhx/ag_news")

In [195]:
# load_dataset returns a datasetDict which containes multiple splits such as: train and test, and it does not have
# .to_pandas() methods. So, you have to convert each split alone. 
print(data)
train_dataset = data["train"]
test_dataset = data["test"]
print(train_dataset[0])
print(test_dataset[0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}
{'text': "Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.", 'label': 2}


### Data Preprocessing 

In [196]:
# Convert the train split to pandas
train_df = data['train'].to_pandas()
test_df = data['test'].to_pandas()

In [197]:
print(train_df.shape)
print("===========")
print(test_df.shape)

(120000, 2)
(7600, 2)


In [198]:
print(train_df.head())
print(test_df.head())

                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2
3  Iraq Halts Oil Exports from Main Southern Pipe...      2
4  Oil prices soar to all-time record, posing new...      2
                                                text  label
0  Fears for T N pension after talks Unions repre...      2
1  The Race is On: Second Private Team Sets Launc...      3
2  Ky. Company Wins Grant to Study Peptides (AP) ...      3
3  Prediction Unit Helps Forecast Wildfires (AP) ...      3
4  Calif. Aims to Limit Farm-Related Smog (AP) AP...      3


In [199]:
train_df.isnull().sum()

text     0
label    0
dtype: int64

In [200]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   text    120000 non-null  str  
 1   label   120000 non-null  int64
dtypes: int64(1), str(1)
memory usage: 28.9 MB


In [201]:
# Training 
X_train = np.array(train_df.iloc[:, 0]) # Training text, its a list of sentences not only one sentence.
print(X_train[0])
y_train = np.array(train_df.iloc[:, 1]) # Training label 
print(y_train[0])

# Testing 
X_test = np.array(test_df.iloc[:, 0]) # Testing text
print(X_test[0])
y_test = np.array(test_df.iloc[:, 1]) # Testing label
print(y_test[0])

Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
2
Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.
2


In [202]:
# The data is already splitted into training and testing.
# But we need to split the testing dataset into validation data and testing data.

# Split the test data into testing and validation
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5)

#### LSTM  cannot read English, it only understand numbers, so we will have to turn the words into numbers.

#### Step 1. Building the Vocabulary

- Vocabulary: simply a unique dictionary of all the words your model knows.
1. You read all the training sentences once. 
2. You write down every unique word.
3. Assign each word a number.
- Why does vocabulary need to be unique? Because the model needs a mapping from words to numbers.

#### Step 2. Tokenization

- A token is simply one unit of text, usually a token is one word. 
- The process of splitting text into tokens is called **tokenization**.
- There are different types of tokenization, like: word tokenization or character tokenization.
- In **Step 1** we have the words as individuals, and the vocabulary dictionary is words not sentences, so we must first split the sentences into words i.e. tokenization.
- PyTorch can not convert raw text into tensors, it should be converted into nubmers first, then tokenized.

#### Step 3. Convert Tokens into IDs

- Now we combine the first two steps. Replace every word using the vocabulary. Now a sentence becomes like that:
[0,1,2,3]

- So we will use the dictionary (translation table) to replace each word with its ID.


#### What about words we have never seen them before:

##### For example: Tesla.

- It does not exits in the vocabulary. 
- We create a special token called UNK --> Unknown word. 
- We give it a unique number (ID).

#### Step 4. Padding 

- When training, every sentence has a different length. For example: [3,5,8] and [6,10,8,9,2,0,1].
- A batch simply contain multiple training examples the model processes together (sentences in your case). 
- Neural networks process data in batches, this is way much faster for the model.
- PyTorch stores the batch in a single tenosr. 
- A tensor requires every row to have the same length (rectangular tensors). 
- Since sentences naturally have different lengths they will not be rectangular tensors like this:
- [
  
    [3, 5, 8],

    [6, 10, 2, 8, 1, 3, 9]

]

- Padding fixes this:
    1. We choose a maximum length.
    2. Add a special value called PAD until it reaches length 7 for sentences that are less then 7.
    3. Usually PAD = 0.
    4. When both sentences are have the same length, Now PyTorch can make one tensor. 
    5. Now the **DataLoader** can give this whole batch to the LSTM at once.
- If the sentence is too long we cut it down using **Truncation**.

In [203]:
# Step 1: Building the tokeniztion list.

# The corpus is all articles, which is X_train. 

# When creating a vocabulary, we want to have a list of words, not a list of lists. 
# append() adds sentence by sentence (which will create a list (the vocabulary) of lists (sentences of corpus)).
# extend() when we want to merge a another collection into a list. so it will add the sentences without being a list.  
# For NLP its prefered to use extend() to create the vocabulary. 

words = [] # A list containign all tokens from the training set. You can call it tokenized corpus.

for sentence in X_train:
    word = sentence.split() # Tokenization
    words.extend(word)

print(words[:45])
print(len(words))

['Wall', 'St.', 'Bears', 'Claw', 'Back', 'Into', 'the', 'Black', '(Reuters)', 'Reuters', '-', 'Short-sellers,', 'Wall', "Street's", 'dwindling\\band', 'of', 'ultra-cynics,', 'are', 'seeing', 'green', 'again.', 'Carlyle', 'Looks', 'Toward', 'Commercial', 'Aerospace', '(Reuters)', 'Reuters', '-', 'Private', 'investment', 'firm', 'Carlyle', 'Group,\\which', 'has', 'a', 'reputation', 'for', 'making', 'well-timed', 'and', 'occasionally\\controversial', 'plays', 'in', 'the']
4541694


In [204]:
# Step 2: Building the vocabulary

# A vocabulary is the unique set of words your model knows.
# As the tokens list is now have a lot of duplicates, we need to remove them so we can map each word to its ID.
# This is called Vocabulary mapping or Word-to-index dictionary. 

unique_vocabulary = set(words) # unique vocab


In [205]:
# set --> list

# Now we need to convert the set into a list, because a set has not fiex order.
# We need a fixed order. so that when we train the model again or save it or debug it it will be easier.

vocabulary = list(unique_vocabulary)
vocabulary = ["<PAD>", "<UNK>"] + vocabulary
# These are fixed:
# <PAD> ---> 0
# <UNK> ---> 1

In [206]:
len(vocabulary)

188112

In [207]:
# ID mapping

# The goal is to create dictionary that maps each words into a unique ID.
# the method enumerate() gives you 2 things when used with a list: 
# 1- The index
# 2- The actual value (word) 

# (index, word)

# (0, "i")
# (1, "love")
# (2, "ai")
# (3, "python")

word_to_id = {}

for index, word in enumerate(vocabulary):
    word_to_id[word] = index




In [208]:
# Numericalization

# In this step, we want to convert all sentences into numbers according to the ID map dictionary.
# Before we had only a word --> ID, we want the whole sentence to be a numbers not just one word. 

def encode_sentence(sentence, word_to_id):
    tokens = sentence.split()
    # [the, oil]

    ids = []


    # It will take the word like: oil, then it will look for the coresponding ID and get it and then append it.
    # Will do this for all words in the sentence and then it will return the numbers array.
    for word in tokens:
        ids.append(word_to_id.get(word, word_to_id["<UNK>"])) 
        # If the word exits in the vocabulary (which is built by using the training dataset) get the ID of that
        # word else return the ID of unknown word which is 1 (fixed)

    return ids



# For all sentences in the training set

X_train_ids = [] # This list will now have: [ [5575, 2875, 2542] --> sentence 1, [654, 1246, 756] --> sentence 2, ... ]

X_val_ids = [] # Also the validation set need to be encoded with the same vocabulary, so the model do not get 
# get confused if we used other vocabulary.

X_test_ids = [] # Also the training set need to be encoded

for sentence in X_train:
    ids = encode_sentence(sentence, word_to_id)
    X_train_ids.append(ids)

for sentence in X_val:
    ids = encode_sentence(sentence, word_to_id)
    X_val_ids.append(ids)

for sentence in X_test:
    ids = encode_sentence(sentence, word_to_id)
    X_test_ids.append(ids)



In [209]:
print(encode_sentence("the apple oil", word_to_id))

[93249, 112488, 50744]


In [210]:
# Padding sequences

# Every sentence has a different lenght. So we need to make them with same length for regtangle tensor.

# Manually part 



# max_length = 6

# sequences = [
#     [5, 10, 22],
#     [4, 8, 15, 9, 33, 50]
# ]

# padded_sequnces = []

# for sequence in sequences:

#     padding_length = max_length - len(sequence)

#     padded_seq = sequence + [0] * padding_length # [0] * 3 --> [0,0,0]

#     padded_sequnces.append(padded_seq)



max_length = 100

X_train_padded = [] # After padding they will be saved here 
X_test_padded = []
X_val_padded = []

PAD_ID = word_to_id["<PAD>"] # get the pad id from the vocabulary 

for sentence in X_train_ids:

    # If sentence is longer, cut it
    sentence = sentence[:max_length] # The colon : means "go from the very start (index 0) up to, but not including, the index max_length".

    # If sentence is shorter, add padding 

    padding_length = max_length - len(sentence)

    sentence = sentence + [PAD_ID] * padding_length

    X_train_padded.append(sentence)

for sentence in X_test_ids:

    # If sentence is longer, cut it
    sentence = sentence[:max_length] # The colon : means "go from the very start (index 0) up to, but not including, the index max_length".

    # If sentence is shorter, add padding 

    padding_length = max_length - len(sentence)

    sentence = sentence + [PAD_ID] * padding_length

    X_test_padded.append(sentence)


for sentence in X_val_ids:

    # If sentence is longer, cut it
    sentence = sentence[:max_length] # The colon : means "go from the very start (index 0) up to, but not including, the index max_length".

    # If sentence is shorter, add padding 

    padding_length = max_length - len(sentence)

    sentence = sentence + [PAD_ID] * padding_length

    X_val_padded.append(sentence)


# Why padding the input (x) and not also padding y (labels) ? because X = the sentence and LTSM needs it to be a tensor 
# but y the label is a single class, there is no variable length. 


In [211]:
X_train_padded = np.array(X_train_padded)
print(X_train_padded.shape)

print(len(X_train_padded[0]))
print(len(X_val_padded[1]))
print(len(X_test_padded[100]))

(120000, 100)
100
100
100


### Preparing Data 

In [212]:
# Convert integers to tensors

X_train_tensor = torch.tensor(X_train_padded, dtype=torch.long)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_padded, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

X_vali_tensor = torch.tensor(X_val_padded, dtype=torch.long)
y_vali_tensor = torch.tensor(y_val, dtype=torch.long)

- Dataset makes shuffling and getting specific samples and splitting the data into batches easier.
- Dataset Stores data and fetches items by index.
- The Dataset provides the samples, and the DataLoader groups them into batches.

#### Training Process with Batches

### Batch 1
* Samples: 0–31
  * ↓
* Model
  * ↓
* Loss
  * ↓
* Update weights

---

### Batch 2
* Samples: 32–63
  * ↓
* Model
  * ↓
* Loss
  * ↓
* Update weights

1. **Fetch Batch 1** (Samples 0–31)
2. **Forward Pass** → Pass samples through the model
3. **Calculate Loss** → Measure prediction error
4. **Update Weights** → Adjust weights via gradient descent
5. **Fetch Batch 2** (Samples 32–63)
6. **Repeat Process** → Pass new batch through updated model

In [213]:
class AGNewsDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]

In [214]:
training_data = AGNewsDataset(X_train_tensor, y_train_tensor)
validation_data = AGNewsDataset(X_vali_tensor, y_vali_tensor)
testing_data = AGNewsDataset(X_test_tensor, y_test_tensor)

In [215]:
train_dataloader = DataLoader(training_data, batch_size=32, shuffle=True)
validation_dataloader = DataLoader(validation_data, batch_size=32, shuffle=True)
test_dataloader = DataLoader(testing_data, batch_size=32, shuffle=True)

### LTSM

1. **Input IDs**
   * ↓
2. **Embedding Layer**
   * ↓
3. **LSTM Layer**
   * ↓
4. **Linear Layer**
   * ↓
5. **Class Prediction**

- After we have done the numericalization and we get an array of numbers for each sentence, these numbers still does not have meaning yet. The model does not know 82 --> "market" or whatever.
  
- To solve this, we use the embedding layer. It learns a vector representation for every word. 
  
- Before: "market" → 82. After embeddings 82 → [0.21, -0.53, 0.77, 0.15]. Now the model can learn relationships between words. 

- The embedding layer needs: 
  * 1- Vocabulary size (How many unique words exist?) 
  * 2- Embedding dimension (How many numbers represent each word?). Example: 50000 words --> 50000 vectors (each vector has 100 values / dimensions).

- Now the words of a sentence are vectors. 

Sentence:

"The company increased profits"


After embedding:

[
 * vector for "The",
 * vector for "company",
 * vector for "increased",
 * vector for "profits"
  
]

The LSTM reads these vectors one by one.

It maintains a memory called: **hidden state**

The idea:
1. **"The"** 
   * ↓
   * **memory updated**
2. **"company"** 
   * ↓
   * **memory updated**
3. **"increased"** 
   * ↓
   * **memory updated**
4. **"profits"** 
   * ↓
   * **final understanding**

- In LSTM we do not have a hidden neurons like neural networksl we have **hidden size**. It controls the size of the LSTM memroy size.
  
- Example: hidden_size = 10, means:

"The LSTM stores the sentence information using 10 numbers.", The more you increase the hidden size the more capacity to remember patterns.

- The LSTM produces a final hidden representation. Example: LSTM output: 
[
0.4,
0.8,
-0.2,
...
]

- Then the linear layer (classifier). It converts that into a class score. 

- We have 
* 0 → World
* 1 → Sports
* 2 → Business
* 3 → Sci/Tech

So it must produce 4 numbers, each with a score, and the biggest will be the answer. 


In [216]:
# Note: number of embeddings = number of words / size of vocabulary 

HIDDEN_SIZE = 10 # LSTM memory size
EMBEDDING_DIMENSION = 100 # How many numbers represents each word
NUM_CLASSES = 4

class LSTMClassifier(nn.Module):

    def __init__(self, vocabulary_size):
        super(LSTMClassifier, self).__init__()

    # We need to create 3 layers: Embeddign layer. LSTM layer. Linear layer

        # Convert word IDs into word vector
        self.embedding_layer = nn.Embedding(num_embeddings=vocabulary_size, embedding_dim=EMBEDDING_DIMENSION)

        # LSTM layer
        self.lstm_layer = nn.LSTM(input_size=EMBEDDING_DIMENSION, hidden_size=HIDDEN_SIZE, batch_first=True)
        # input_size = How many numbers does the LSTM receive for each word? word / embedding dimension
        # hidden_size = LSTM memory size, The LSTM summarizes the sentence into 10 numbers
        # batch_first = only controls the input shape

        # Classifier layer
        self.classifier_layer = nn.Linear(HIDDEN_SIZE, NUM_CLASSES)
        # It will convert the 10 numbers from the hidden size into 4 for the NUM_CLASSES


    # The data flow
    def forward(self, x):

        embedded = self.embedding_layer(x) # Create the embeddings 

        output, (hidden, cell) = self.lstm_layer(embedded)
        # When the LSTM reads: word 1,2 ,3 ,4 it produces: output, hidden, cell 
        # hidden = the summary / shape of the sentence. (number of layers, batch, hidden size)


        hidden = hidden[-1] # -1 --> We remove the layer dimension. Now each sentence has a 10-number representation.

        prediction = self.classifier_layer(hidden)

        return prediction

        

In [217]:
model = LSTMClassifier(vocabulary_size=len(vocabulary)).to(device)

In [218]:
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=1e-3)

In [226]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
# Training and validatoin 

epochs = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

# Training 
for epoch in range(epochs):

    print(f"Epoch {epoch+1}/{epochs}")

    model.train()

    total_acc_train = 0
    total_loss_train = 0
    total_acc_val = 0
    total_loss_val = 0

    for data in train_dataloader:

        for i, data in enumerate(train_dataloader):
            if i % 100 == 0:
                print(f"Batch {i}")

        inputs, labels = data

        inputs = inputs.to(device)
        labels = labels.to(device)

        prediction = model(inputs)
        batch_loss = criterion(prediction, labels)


        total_loss_train += batch_loss.item()

        # Get the predicted class for each sample 
        predicted_train_class = torch.argmax(prediction, dim=1)

        acc = (predicted_train_class == labels).sum().item()

        total_acc_train += acc


        # The model learning 

        batch_loss.backward()

        optimizer.step()

        optimizer.zero_grad()


    with torch.no_grad():

        model.eval()
        

        for data in validation_dataloader:

            inputs, labels = data

            inputs = inputs.to(device)
            labels = labels.to(device)

            prediction = model(inputs)

            batch_loss = criterion(prediction, labels)

            total_loss_val += batch_loss.item()

            predicted_val_class = torch.argmax(prediction, dim=1)

            acc = (predicted_val_class == labels).sum().item()

            total_acc_val += acc



In [ ]:
# Testing 

with torch.no_grad():

        total_loss_test = 0
        total_acc_test = 0
        

        for data in test_dataloader:

            inputs, labels = data

            inputs = inputs.to(device)
            labels = labels.to(device)

            prediction = model(inputs)

            batch_loss = criterion(prediction, labels)

            total_loss_val += batch_loss.item()

            predicted_val_class = torch.argmax(prediction, dim=1)

            acc = (predicted_val_class == labels).sum().item()

            total_acc_val += acc
            
print("Accuracy: ", round(total_acc_test / testing_data.__len__() * 100, 4))

In [ ]:
# Inferene 

# Classes in AG News
classes = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}


def predict(text):

    model.eval()

    # Tokenization
    tokens = text.lower().split()

    # Convert words to indices using your vocabulary
    token_ids = []

    for word in tokens:
        if word in vocab:
            token_ids.append(vocab[word])
        else:
            token_ids.append(vocab["<UNK>"])


    # Padding (same length used during training)
    max_length = 100  # change this to your padding length

    if len(token_ids) < max_length:
        token_ids += [vocabulary["<PAD>"]] * (max_length - len(token_ids))
    else:
        token_ids = token_ids[:max_length]


    # Convert to tensor
    input_tensor = torch.tensor(token_ids, dtype=torch.long)

    # Add batch dimension
    # From:
    # (sequence_length)
    #
    # To:
    # (batch_size, sequence_length)
    input_tensor = input_tensor.unsqueeze(0)


    # Move to GPU if available
    input_tensor = input_tensor.to(device)


    # Prediction
    with torch.no_grad():

        output = model(input_tensor)

        predicted_class = torch.argmax(output, dim=1).item()


    return classes[predicted_class]

text = """
Apple announced a new AI chip for its computers today.
"""

prediction = predict(text)

print(prediction)
